<a href="https://colab.research.google.com/github/AndreYuanAngPSHSSMCACCOUNT3/Final-Project-CS2/blob/main/Jasmine_Hospital_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import firebase_admin
from firebase_admin import credentials

cred = credentials.Certificate("/content/jasmine-hospital-db-firebase-adminsdk-fbsvc-dd4f299ccd.json")

# Initialize Firebase only if it hasn't been initialized already
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

if firebase_admin._apps:
    print(f"✅ Firebase is connected! Project ID: {firebase_admin.get_app().project_id}")
else:
    print("❌ Firebase is not initialized.")

✅ Firebase is connected! Project ID: jasmine-hospital-db


In [ ]:
import pandas as pd
import json
import os

# Define the file name for the hospital data
FILE_NAME = "hospital.json"

def load_patients_as_dict():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, 'r') as f:
            return json.load(f)
    return {}

# Load the data from the JSON file into a dictionary
patients_data = load_patients_as_dict()

# Convert the dictionary of patients to a list of patient records
# Each patient record should be a dictionary that can be flattened into a DataFrame row
patient_records = []
# Iterate directly over the list of patient dictionaries
for patient_details in patients_data:
    record = {
        'patient_id': patient_details.get('patient_id'),
        'name': patient_details.get('name'),
        'visits': patient_details.get('visits', []) # Ensure 'visits' is a list, even if empty
    }
    patient_records.append(record)

# Create the DataFrame df_hospital
df_hospital = pd.DataFrame(patient_records)


#Feature 1, by King Sabas
#This feature displays the patient name, their ID, and how many times they visited
print("--- Patient Visit Counts ---")
if not df_hospital.empty:
    for index, patient_row in df_hospital.iterrows():
        patient_id = patient_row['patient_id']
        patient_name = patient_row['name']
        number_of_visits = len(patient_row['visits'])
        print(f"Patient Name: {patient_name} (ID: {patient_id}) visited {number_of_visits} times.")
else:
    print("No patient data available to display visit counts.")

--- Patient Visit Counts ---
Patient Name: John Cruz (ID: 1) visited 2 times.
Patient Name: Ana Gomez (ID: 2) visited 2 times.
Patient Name: Luis Ramos (ID: 3) visited 2 times.
Patient Name: Maria Santos (ID: 4) visited 2 times.


In [ ]:
#Feature 2, by Lawrence Perez
#This feature displays the doctor the patients have consulted
print("--- Doctors Consulted by Patient ---")
for index, patient_row in df_hospital.iterrows():
    patient_id = patient_row['patient_id']
    patient_name = patient_row['name']
    doctors_seen = set()
    for visit in patient_row['visits']:
        if isinstance(visit, dict) and 'doctor' in visit:
            doctors_seen.add(visit['doctor'])
    print(f"Patient Name: {patient_name} (ID: {patient_id}) consulted: {', '.join(doctors_seen) if doctors_seen else 'None'}.")

--- Doctors Consulted by Patient ---
Patient Name: John Cruz (ID: 1) consulted: Dr. Santos.
Patient Name: Ana Gomez (ID: 2) consulted: Dr. Dela Cruz.
Patient Name: Luis Ramos (ID: 3) consulted: Dr. Reyes.
Patient Name: Maria Santos (ID: 4) consulted: Dr. Tan.


In [ ]:
#Feature 3, by Mohammad Madki
#This feature displays the patient with the most visits
print("-- Patient with Most Visits ---")
# First, let's collect visit counts for all patients
all_patient_visits = []
for index, patient_row in df_hospital.iterrows():
    patient_id = patient_row['patient_id']
    patient_name = patient_row['name']
    number_of_visits = len(patient_row['visits'])
    all_patient_visits.append({'patient_id': patient_id, 'name': patient_name, 'visits_count': number_of_visits})

# Now find the patient who visited the most
if all_patient_visits:
    patient_with_most_visits = max(all_patient_visits, key=lambda x: x['visits_count'])
    print(f"The patient who visited the hospital the most times is {patient_with_most_visits['name']} (ID: {patient_with_most_visits['patient_id']}) with {patient_with_most_visits['visits_count']} visits.")
else:
    print("No patient data available to find the one with most visits.")

-- Patient with Most Visits ---
The patient who visited the hospital the most times is John Cruz (ID: 1) with 2 visits.


In [ ]:
import pandas as pd
import json
import os
from collections import Counter
import time
import firebase_admin
from firebase_admin import firestore

# Define the file name for the hospital data
FILE_NAME = "hospital.json"

def load_patients_for_summary_data():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, 'r') as f:
            data = json.load(f)
            # If data is a list, convert it to a list of dicts for DataFrame
            if isinstance(data, list):
                return data
            # If it's a dict of dicts, convert values to a list of dicts
            return list(data.values())
    return []

def load_patients_for_crud_operations():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, 'r') as f:
            data = json.load(f)
            # If data is a list, convert it to a dictionary keyed by patient_id
            if isinstance(data, list):
                return {str(p['patient_id']): p for p in data}
            return data # Assume it's already a dictionary if not a list
    return {}

def save_patients_for_crud_operations(data):
    with open(FILE_NAME, 'w') as f:
        # Convert dictionary values back to a list for saving to JSON file
        json.dump(list(data.values()), f, indent=4)

    # Also save to Firebase Firestore
    if firebase_admin._apps:
        db = firestore.client()
        for patient_id_str, patient_data in data.items():
            doc_ref = db.collection('patients').document(patient_id_str)
            try:
                doc_ref.set(patient_data)
            except Exception as e:
                print(f"Error saving patient {patient_id_str} to Firestore: {e}")

# Load the data from the JSON file into a list of patient records for summary
patients_data_for_summary = load_patients_for_summary_data()

patient_records_for_summary = []
for patient_details in patients_data_for_summary:
    record = {
        'patient_id': patient_details.get('patient_id'),
        'name': patient_details.get('name'),
        'visits': patient_details.get('visits', []) # Ensure 'visits' is a list, even if empty
    }
    patient_records_for_summary.append(record)

df_hospital_summary = pd.DataFrame(patient_records_for_summary)

# Prepare patient summaries once
patient_summaries = []
all_doctors_in_visits = []
all_diagnoses_in_visits = []

if not df_hospital_summary.empty:
    for index, patient_row in df_hospital_summary.iterrows():
        patient_id = patient_row['patient_id']
        patient_name = patient_row['name']
        patient_visits = patient_row['visits']

        # Feature 1: Count how many times a patient visited
        number_of_visits = len(patient_visits)

        # Feature 2: List the doctors a patient saw
        doctors_seen = set()
        for visit in patient_visits:
            if isinstance(visit, dict):
                if 'doctor' in visit:
                    doctors_seen.add(visit['doctor'])
                    all_doctors_in_visits.append(visit['doctor'])
                if 'diagnosis' in visit:
                    all_diagnoses_in_visits.append(visit['diagnosis'])

        # Store all this info for the patient
        patient_summaries.append({
            'patient_id': patient_id,
            'name': patient_name,
            'visits_count': number_of_visits,
            'doctors_consulted': list(doctors_seen)
        })

# --- Calculations for new features ---
doctor_visit_counts = Counter(all_doctors_in_visits)
most_visited_doctor_info = None
if doctor_visit_counts:
    most_visited_doctor = doctor_visit_counts.most_common(1)[0]
    most_visited_doctor_info = f"{most_visited_doctor[0]} with {most_visited_doctor[1]} visits."

average_visits_per_doctor = 0
if doctor_visit_counts:
    total_visits_across_all_doctors = sum(doctor_visit_counts.values())
    num_unique_doctors = len(doctor_visit_counts)
    if num_unique_doctors > 0:
        average_visits_per_doctor = total_visits_across_all_doctors / num_unique_doctors

diagnosis_counts = Counter(all_diagnoses_in_visits)
most_common_diagnosis_info = None
if diagnosis_counts:
    most_common_diagnosis = diagnosis_counts.most_common(1)[0]
    most_common_diagnosis_info = f"{most_common_diagnosis[0]} (occurred {most_common_diagnosis[1]} times).";

# Encapsulate the summary menu into a function
def run_summary_menu():
    while True:
        print("\n===== PATIENT DATA SUMMARY MENU ====")
        print("1. Display All Patient Visit Details")
        print("2. List Doctors Consulted by Patient")
        print("3. Display Patient with Most Visits")
        print("4. Find the Most Visited Doctor")
        print("5. Compute Average Visits Per Doctor")
        print("6. Show Most Common Diagnosis")
        print("7. Exit Summary")

        choice = input("Enter your choice: ")

        if choice == "1":
            print("\n--- Patient Visit Details ---")
            if patient_summaries:
                for summary in patient_summaries:
                    print(f"\nPatient ID: {summary['patient_id']}")
                    print(f"Name: {summary['name']}")
                    print(f"Number of visits: {summary['visits_count']}")
                    print(f"Doctors consulted: {', '.join(summary['doctors_consulted']) if summary['doctors_consulted'] else 'None'}")
            else:
                print("No patient data available for display.")

        elif choice == "2":
            print("\n--- Doctors Consulted by Patient ---")
            if patient_summaries:
                for summary in patient_summaries:
                    print(f"Patient Name: {summary['name']} (ID: {summary['patient_id']}) consulted: {', '.join(summary['doctors_consulted']) if summary['doctors_consulted'] else 'None'}.")
            else:
                print("No patient data available to list doctors.")

        elif choice == "3":
            print("\n--- Who visited the most? ---")
            if patient_summaries:
                patient_with_most_visits = max(patient_summaries, key=lambda x: x['visits_count'])
                print(f"The patient who visited the hospital the most times is {patient_with_most_visits['name']} (ID: {patient_with_most_visits['patient_id']}) with {patient_with_most_visits['visits_count']} visits.")
            else:
                print("No patient data available to find the one with most visits.")

        elif choice == "4":
            print("\n--- Most Visited Doctor ---")
            if most_visited_doctor_info:
                print(f"The most visited doctor is: {most_visited_doctor_info}")
            else:
                print("No doctor visit data available.")

        elif choice == "5":
            print("\n--- Average Visits Per Doctor ---")
            if average_visits_per_doctor > 0:
                print(f"The average number of visits per doctor is: {average_visits_per_doctor:.2f}")
            else:
                print("No doctor visit data available to compute average.")

        elif choice == "6":
            print("\n--- Most Common Diagnosis ---")
            if most_common_diagnosis_info:
                print(f"The most common diagnosis is: {most_common_diagnosis_info}")
            else:
                print("No diagnosis data available.")

        elif choice == "7":
            print("Exiting Summary Menu.")
            return # Use return to go back to the calling menu

        else:
            print("Invalid choice. Please try again.")


In [ ]:
import json
import os
import time
import firebase_admin
from firebase_admin import firestore
import pandas as pd
from collections import Counter

# Define the file name for the hospital data
FILE_NAME = "hospital.json"

def load_patients_for_summary_data():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, 'r') as f:
            data = json.load(f)
            # If data is a list, convert it to a list of dicts for DataFrame
            if isinstance(data, list):
                return data
            # If it's a dict of dicts, convert values to a list of dicts
            return list(data.values())
    return []

def load_patients_for_crud_operations():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, 'r') as f:
            data = json.load(f)
            # If data is a list, convert it to a dictionary keyed by patient_id
            if isinstance(data, list):
                return {str(p['patient_id']): p for p in data}
            return data # Assume it's already a dictionary if not a list
    return {}

def save_patients_for_crud_operations(data):
    with open(FILE_NAME, 'w') as f:
        # Convert dictionary values back to a list for saving to JSON file
        json.dump(list(data.values()), f, indent=4)

    # Also save to Firebase Firestore
    if firebase_admin._apps:
        db = firestore.client()
        for patient_id_str, patient_data in data.items():
            doc_ref = db.collection('patients').document(patient_id_str)
            try:
                doc_ref.set(patient_data)
            except Exception as e:
                print(f"Error saving patient {patient_id_str} to Firestore: {e}")

# Load the data from the JSON file into a list of patient records for summary
patients_data_for_summary = load_patients_for_summary_data()

patient_records_for_summary = []
for patient_details in patients_data_for_summary:
    record = {
        'patient_id': patient_details.get('patient_id'),
        'name': patient_details.get('name'),
        'visits': patient_details.get('visits', []) # Ensure 'visits' is a list, even if empty
    }
    patient_records_for_summary.append(record)

df_hospital_summary = pd.DataFrame(patient_records_for_summary)

# Prepare patient summaries once
patient_summaries = []
all_doctors_in_visits = []
all_diagnoses_in_visits = []

if not df_hospital_summary.empty:
    for index, patient_row in df_hospital_summary.iterrows():
        patient_id = patient_row['patient_id']
        patient_name = patient_row['name']
        patient_visits = patient_row['visits']

        # Feature 1: Count how many times a patient visited
        number_of_visits = len(patient_visits)

        # Feature 2: List the doctors a patient saw
        doctors_seen = set()
        for visit in patient_visits:
            if isinstance(visit, dict):
                if 'doctor' in visit:
                    doctors_seen.add(visit['doctor'])
                    all_doctors_in_visits.append(visit['doctor'])
                if 'diagnosis' in visit:
                    all_diagnoses_in_visits.append(visit['diagnosis'])

        # Store all this info for the patient
        patient_summaries.append({
            'patient_id': patient_id,
            'name': patient_name,
            'visits_count': number_of_visits,
            'doctors_consulted': list(doctors_seen)
        })

# --- Calculations for new features ---
doctor_visit_counts = Counter(all_doctors_in_visits)
most_visited_doctor_info = None
if doctor_visit_counts:
    most_visited_doctor = doctor_visit_counts.most_common(1)[0]
    most_visited_doctor_info = f"{most_visited_doctor[0]} with {most_visited_doctor[1]} visits."

average_visits_per_doctor = 0
if doctor_visit_counts:
    total_visits_across_all_doctors = sum(doctor_visit_counts.values())
    num_unique_doctors = len(doctor_visit_counts)
    if num_unique_doctors > 0:
        average_visits_per_doctor = total_visits_across_all_doctors / num_unique_doctors

diagnosis_counts = Counter(all_diagnoses_in_visits)
most_common_diagnosis_info = None
if diagnosis_counts:
    most_common_diagnosis = diagnosis_counts.most_common(1)[0]
    most_common_diagnosis_info = f"{most_common_diagnosis[0]} (occurred {most_common_diagnosis[1]} times)."

# Encapsulate the summary menu into a function
def run_summary_menu():
    while True:
        print("\n===== PATIENT DATA SUMMARY MENU ====")
        print("1. Display All Patient Visit Details")
        print("2. List Doctors Consulted by Patient")
        print("3. Display Patient with Most Visits")
        print("4. Find the Most Visited Doctor")
        print("5. Compute Average Visits Per Doctor")
        print("6. Show Most Common Diagnosis")
        print("7. Exit Summary")

        choice = input("Enter your choice: ")

        if choice == "1":
            print("\n--- Patient Visit Details ---")
            if patient_summaries:
                for summary in patient_summaries:
                    print(f"\nPatient ID: {summary['patient_id']}")
                    print(f"Name: {summary['name']}")
                    print(f"Number of visits: {summary['visits_count']}")
                    print(f"Doctors consulted: {', '.join(summary['doctors_consulted']) if summary['doctors_consulted'] else 'None'}")
            else:
                print("No patient data available for display.")

        elif choice == "2":
            print("\n--- Doctors Consulted by Patient ---")
            if patient_summaries:
                for summary in patient_summaries:
                    print(f"Patient Name: {summary['name']} (ID: {summary['patient_id']}) consulted: {', '.join(summary['doctors_consulted']) if summary['doctors_consulted'] else 'None'}.")
            else:
                print("No patient data available to list doctors.")

        elif choice == "3":
            print("\n--- Who visited the most? ---")
            if patient_summaries:
                patient_with_most_visits = max(patient_summaries, key=lambda x: x['visits_count'])
                print(f"The patient who visited the hospital the most times is {patient_with_most_visits['name']} (ID: {patient_with_most_visits['patient_id']}) with {patient_with_most_visits['visits_count']} visits.")
            else:
                print("No patient data available to find the one with most visits.")

        elif choice == "4":
            print("\n--- Most Visited Doctor ---")
            if most_visited_doctor_info:
                print(f"The most visited doctor is: {most_visited_doctor_info}")
            else:
                print("No doctor visit data available.")

        elif choice == "5":
            print("\n--- Average Visits Per Doctor ---")
            if average_visits_per_doctor > 0:
                print(f"The average number of visits per doctor is: {average_visits_per_doctor:.2f}")
            else:
                print("No doctor visit data available to compute average.")

        elif choice == "6":
            print("\n--- Most Common Diagnosis ---")
            if most_common_diagnosis_info:
                print(f"The most common diagnosis is: {most_common_diagnosis_info}")
            else:
                print("No diagnosis data available.")

        elif choice == "7":
            print("Exiting Summary Menu.")
            return # Use return to go back to the calling menu

        else:
            print("Invalid choice. Please try again.")

def run_patient_database_menu():
    while True:
        print("\n===== PATIENT DATABASE MENU ====")
        print("1. Display Patients")
        print("2. Add Patient")
        print("3. Update Patient (add a new visit)")
        print("4. Delete Patient")
        print("5. Go to Features Menu") # Changed option 5
        print("6. Exit Database Management") # Added option 6 to exit

        choice = input("Enter choice: ")

        # DISPLAY
        if choice == "1":
            start_time = time.time()
            patients = load_patients_for_crud_operations() # Use function from previous cell
            if patients:
                print("\nPatient List:")
                for patient_id_str, patient in patients.items():
                    patient_id = patient['patient_id']
                    patient_name = patient['name']
                    number_of_visits = len(patient['visits']) if 'visits' in patient and patient['visits'] else 0
                    print(f"ID: {patient_id}, Name: {patient_name}, Visits: {number_of_visits}")
            else:
                print("No patients in the database.")
            end_time = time.time()
            print(f"Display operation took {end_time - start_time:.4f} seconds.")

        # ADD PATIENT (with initial visit details)
        elif choice == "2":
            start_time = time.time()
            patients = load_patients_for_crud_operations() # Use function from previous cell
            patient_id_input = input("Enter Patient ID: ")

            # Check if patient ID already exists
            if patient_id_input in patients:
                print(f"Patient with ID {patient_id_input} already exists. Please choose a different ID or update the existing patient.")
                continue

            patient_name_input = input("Enter Patient Name: ")

            new_patient = {
                "patient_id": int(patient_id_input),
                "name": patient_name_input,
                "visits": []
            }

            add_initial_visit = input("Do you want to add an initial visit for this patient? (yes/no): ").lower()
            if add_initial_visit == 'yes':
                diagnosis = input("Enter Diagnosis: ")
                doctor = input("Enter Doctor's Name: ")
                notes = input("Enter Visit Notes: ")
                treatment = input("Enter Treatment: ")
                visit_date = input("Enter Visit Date (YYYY-MM-DD): ")
                new_patient["visits"].append({
                    "diagnosis": diagnosis,
                    "doctor": doctor,
                    "notes": notes,
                    "treatment": treatment,
                    "visit_date": visit_date
                })

            patients[patient_id_input] = new_patient
            save_patients_for_crud_operations(patients) # Use function from previous cell
            print("Patient added successfully!")
            end_time = time.time()
            print(f"Add operation took {end_time - start_time:.4f} seconds.")

        # UPDATE PATIENT (add a new visit)
        elif choice == "3":
            start_time = time.time()
            patients = load_patients_for_crud_operations() # Use function from previous cell
            patient_id_to_update = input("Enter ID of patient to add a visit for: ")

            if patient_id_to_update in patients:
                print(f"Adding new visit for Patient: {patients[patient_id_to_update]['name']} (ID: {patient_id_to_update})")
                diagnosis = input("Enter Diagnosis: ")
                doctor = input("Enter Doctor's Name: ")
                notes = input("Enter Visit Notes: ")
                treatment = input("Enter Treatment: ")
                visit_date = input("Enter Visit Date (YYYY-MM-DD): ")

                new_visit = {
                    "diagnosis": diagnosis,
                    "doctor": doctor,
                    "notes": notes,
                    "treatment": treatment,
                    "visit_date": visit_date
                }
                patients[patient_id_to_update]['visits'].append(new_visit)
                save_patients_for_crud_operations(patients) # Use function from previous cell
                print("Visit added successfully!")
            else:
                print("Patient not found.")
            end_time = time.time()
            print(f"Update operation took {end_time - start_time:.4f} seconds.")

        # DELETE PATIENT
        elif choice == "4":
            start_time = time.time()
            patients = load_patients_for_crud_operations() # Use function from previous cell
            patient_id_to_delete = input("Enter ID to delete: ")

            if patient_id_to_delete in patients:
                del patients[patient_id_to_delete]
                save_patients_for_crud_operations(patients) # Use function from previous cell
                print("Patient deleted successfully!")
            else:
                print("Patient not found.")
            end_time = time.time()
            print(f"Delete operation took {end_time - start_time:.4f} seconds.")

        # Go to Features Menu (calls the summary menu)
        elif choice == "5":
            print("Going to Features Menu.")
            run_summary_menu() # Call the summary menu function from d4419009
            # After run_summary_menu returns, the patient database menu continues its loop

        # Exit Database Management
        elif choice == "6":
            print("Exiting Patient Database Management Menu.")
            break # Exit this loop

        else:
            print("Invalid choice. Try again.")

# Initial call to start the Patient Database Menu
run_patient_database_menu()


===== PATIENT DATABASE MENU ====
1. Display Patients
2. Add Patient
3. Update Patient (add a new visit)
4. Delete Patient
5. Go to Features Menu
6. Exit Database Management

Patient List:
ID: 1, Name: John Cruz, Visits: 2
ID: 2, Name: Ana Gomez, Visits: 2
ID: 3, Name: Luis Ramos, Visits: 2
ID: 4, Name: Maria Santos, Visits: 2
Display operation took 0.0003 seconds.

===== PATIENT DATABASE MENU ====
1. Display Patients
2. Add Patient
3. Update Patient (add a new visit)
4. Delete Patient
5. Go to Features Menu
6. Exit Database Management
